# A categorically organized CAS in Lean

This notebook is the acceptance proof of the CasDsl vertical slice: a
computer algebra system whose **user-facing interfaces are organized by the
mathematical categories where operations first make sense** — not by
implementation classes, and not by backends.

Everything below runs in a persistent Lean 4 kernel
([lean-jupyter-kernel](https://github.com/dzackgarza/lean-jupyter-kernel)).
Three invariants to watch for:

1. **Backend-blind syntax.** You will never see a backend named in an
   expression. Some results below are computed by SageMath through a direct
   typed adapter — the *developer's* routing configuration decides that,
   and `#explain_route` will show it. The mathematics doesn't change.
2. **Category-owned methods.** `factor`, `det`, `annihilator`, `nth` are
   declared on categories; objects receive them by membership and by
   *subcategory inheritance*, never by forwarding code on a leaf class.
3. **Semantic availability ≠ computability.** A method that makes
   mathematical sense stays available even when no implementation route
   exists yet — execution then fails with a *structured capability gap*
   (an auditable developer backlog item), never a fake value and never a
   type error. The final cell demonstrates this deliberately.


## 1 · Trusted arithmetic and assertions

`assert` is an *operational* assertion in the ordinary CAS sense: the
predicate is computed and trusted, with a fourfold outcome
`true | false | unknown | error`. Only `true` lets the cell commit.
No Lean theorem is generated, and no certificate is required — this is a
CAS, not a proof obligation machine.


In [1]:
assert 2 + 3 = 5

Starting Lean worker (/home/dzack/gitclones/lean-cas-dsl)…


1:0: ✓ 2 + 3 = 5


In [2]:
assert 2 + 3 = 0 in ℤ/5

1:0: ✓ 2 + 3 = 0 in ℤ/5


## 2 · Backend-blind factorization

`factor` is declared on the category of factorization-domain elements.
An integer receives it because `EuclideanElems(ℤ) ≤ FactorizationElems(ℤ)`
— the method arrives by **subcategory inheritance through the category
graph**, and is executed by whatever implementation the developer routed.


In [3]:
let n := 360 in ℤ

1:0: n := 360 ∈ ℤ


In [4]:
n.factor()

1:0: 2^3 * 3^2 * 5


2^3 * 3^2 * 5

The expression above never mentioned a backend. The routing that chose one
is developer diagnostics, not mathematics:


In [5]:
#explain_route n.factor()

1:0:   method:        factor
  receiver:      360 ∈ ℤ
  profile entry: EuclideanElems(ℤ)
  availability:  inherited through EuclideanElems(ℤ) ≤ FactorizationElems
  route:         backend sage, op "factor_int", priority 0
  pattern:       element of ℤ


  method:        factor
  receiver:      360 ∈ ℤ
  profile entry: EuclideanElems(ℤ)
  availability:  inherited through EuclideanElems(ℤ) ≤ FactorizationElems
  route:         backend sage, op "factor_int", priority 0
  pattern:       element of ℤ

## 3 · Polynomials, canonical maps, and calling a polynomial

`ℤ ⊆ ℚ` denotes the preferred canonical map — so `map p to ℚ[x]` moves a
polynomial along it without ceremony. `factor` is routed where `p` lives:
ℤ[x] is a UFD, and comparing its factorization with the one in ℚ[x]
below — content and units differ in general — is itself instructive. And a polynomial can simply be
**called**: elaboration inserts evaluation through the preferred compatible
coefficient map. The mathematician writes `q(1)`, as on paper.

`map e to D` means: apply the preferred canonical map into `D` when one is
registered, and fail honestly otherwise. Canonical maps are *preferred
choices*, not necessarily injections — the inclusions `ℕ ⊆ ℤ ⊆ ℚ` are
monomorphisms, while `ℤ → ℤ/n` is the ring quotient, supplied by its
universal property. As of round two these are **registry data**: the
prelude registers them, the engine knows none of these facts, and an
unregistered pair fails with the honest `there is no preferred canonical
map` error — a missing coercion is never widened to a "reasonable"
conversion.

In [6]:
let p(x) := x^3 - 2x + 1 in ℤ[x]

1:0: p := x^3 - 2x + 1 ∈ ℤ[x]


In [7]:
p.factor()

1:0: (x - 1) * (x^2 + x - 1)


(x - 1) * (x^2 + x - 1)

In [8]:
let q := map p to ℚ[x]

1:0: q := x^3 - 2x + 1 ∈ ℚ[x]


In [9]:
q.factor()

1:0: (x - 1) * (x^2 + x - 1)


(x - 1) * (x^2 + x - 1)

In [10]:
assert q(1) = 0

1:0: ✓ q(1) = 0


The quotient `ℤ → ℤ/n` is one registered rule for *every* modulus: an
integer names its residue class. `n` is still `360`, and `360 ≡ 3 (mod 7)`:

In [11]:
map n to ℤ/7

1:0: 3


3

The coercion surface is itself auditable: these are the *only* maps
the surface will ever insert, straight from the registry (#9):

In [12]:
#canonical_maps

1:0:   map         op
  ℕ → ℤ       identity
              ℕ ⊆ ℤ: an element of ℕ already IS an integer (they share the `Value.int` representation), so the injection moves no data
  ℕ → ℚ       intToRat
              ℕ ⊆ ℚ: the composite of ℕ ⊆ ℤ ⊆ ℚ, registered explicitly because the coercion layer takes ONE hop (it does not compose rules)
  ℤ → ℚ       intToRat
              ℤ ⊆ ℚ: the fraction field of ℤ — the notebook's `map p to ℚ[x]` is this rule applied coefficient-wise
  ℤ → ℤ/_     intToMod
              ℤ → ℤ/n for EVERY modulus n (one rule, by `anyMod`): the ring quotient — an integer naming its residue class, which is what an ascription such as `let x := 7 in ℤ/5` inserts


  map         op
  ℕ → ℤ       identity
              ℕ ⊆ ℤ: an element of ℕ already IS an integer (they share the `Value.int` representation), so the injection moves no data
  ℕ → ℚ       intToRat
              ℕ ⊆ ℚ: the composite of ℕ ⊆ ℤ ⊆ ℚ, registered explicitly because the coercion layer takes ONE hop (it does not compose rules)
  ℤ → ℚ       intToRat
              ℤ ⊆ ℚ: the fraction field of ℤ — the notebook's `map p to ℚ[x]` is this rule applied coefficient-wise
  ℤ → ℤ/_     intToMod
              ℤ → ℤ/n for EVERY modulus n (one rule, by `anyMod`): the ring quotient — an integer naming its residue class, which is what an ascription such as `let x := 7 in ℤ/5` inserts

## 4 · Exact matrix algebra

Matrix literals use row-semicolon syntax; `det` and `inverse` are methods
of the square-matrix category, computed exactly over `ℚ`.


In [13]:
let M := [1, 2; 3, 4] in Mat₂(ℚ)

1:0: M := [1, 2; 3, 4] ∈ Mat₂(ℚ)


In [14]:
M.inverse()

1:0: [-2, 1; 3/2, -1/2]


[-2, 1; 3/2, -1/2]

In [15]:
assert M.det() = -2

1:0: ✓ M.det() = -2


## 5 · Subcategory inheritance, for real

`annihilator : Modules(ℤ) → Ideals(ℤ)` is declared **once**, on the parent
category. `F` below is declared in the *proper subcategory*
`SmallModules(ℤ)` — which contains **no forwarding declaration**. The
method arrives purely through the registered inclusion
`SmallModules ≤ Modules`. (The ascription is doing real semantic work:
`ℤ/4` *in a module category* means the ℤ-module ℤ/4, not the ring.)


In [16]:
let F := ℤ/4 in SmallModules(ℤ)

1:0: F := ℤ/4 as ℤ-module


In [17]:
F.annihilator()

1:0: (4)


(4)

## 6 · Transport along preferred functors

`cardinality` is declared on `Sets` — and a module is not a set. But the
prelude registers the forgetful functor `UnderlyingSet : Modules(ℤ) → Sets`
as *preferred*, so the resolver transports the **receiver**:
`F.cardinality()` resolves as `UnderlyingSet(F).cardinality()`. Nothing was
declared on modules, no forwarding method exists anywhere, and the ordinary
call syntax is unchanged.

In [18]:
F.cardinality()

1:0: 4


4

Membership transports the same way — `2` names a residue class of the
underlying set:

In [19]:
assert 2 ∈ F

1:0: ✓ 2 ∈ F


Equality, by contrast, is **category-bound**. `U(F) = {0, 1, 2, 3}` in
Sets — but `F` itself is a module, and there is no *unique* module
structure on that set, so bare `=` between objects of different
categories is trivially false: it never inserts the functor. Comparing
them requires explicitly asking the question in a common comparison
category — which is exactly what the Sets method `set_eq` does (its
receiver transports, like `∈` above):

In [20]:
assert F ≠ {0, 1, 2, 3}

1:0: ✓ F ≠ {0, 1, 2, 3}


In [21]:
F.set_eq({0, 1, 2, 3})

1:0: true


true

Two guarantees, both machine-checked in the build:

- transport runs **only where direct resolution finds nothing** —
  `annihilator` above still arrives untransported through
  `SmallModules(ℤ) ≤ Modules(ℤ)`, so registering a functor can never take
  a method away from an object that already had it;
- two applicable functors would be an honest *ambiguity error* naming both,
  never a silent pick.

The transport step itself is developer diagnostics, not mathematics:

In [22]:
#explain_route F.cardinality()

1:0:   method:        cardinality
  receiver:      ℤ/4 as ℤ-module
  transport:     functor UnderlyingSet : Modules → Sets
  image:         {0, 1, 2, 3}
  profile entry: FiniteSets(ℤ/4)
  availability:  inherited through FiniteSets(ℤ/4) ≤ CountableSets ≤ Sets
  route:         backend native, op "cardinality", priority 0
  pattern:       any set


  method:        cardinality
  receiver:      ℤ/4 as ℤ-module
  transport:     functor UnderlyingSet : Modules → Sets
  image:         {0, 1, 2, 3}
  profile entry: FiniteSets(ℤ/4)
  availability:  inherited through FiniteSets(ℤ/4) ≤ CountableSets ≤ Sets
  route:         backend native, op "cardinality", priority 0
  pattern:       any set

## 7 · Countable sets, ellipses, and indexing

Countability is mathematical structure — a monomorphism into ℕ — not a
backend capability. A *registered enumeration choice* labels elements, so
countable objects support `nth` (`X[k]`, 0-based) and `cardinality`.
Ellipsis literals are the exact Haskell-style progressions, nothing more.

The registered convention for `ℤ` is `0, 1, −1, 2, −2, …` — a documented,
revisitable choice, never a claim that ℤ is intrinsically ordered that way.


In [23]:
let X := {0, 1, 2, ...}

1:0: X := {0, 1, ...}


In [24]:
assert X = ℕ

1:0: ✓ X = ℕ


In [25]:
let Y := {0, 2, 4, ...}

1:0: Y := {0, 2, ...}


In [26]:
assert 8 ∈ Y

1:0: ✓ 8 ∈ Y


In [27]:
assert 9 ∉ Y

1:0: ✓ 9 ∉ Y


In [28]:
ℤ[3]

1:0: 2


2

`ℚ` indexes by *its* registered convention too — the Cantor zigzag
(`0, 1, −1, 1/2, −1/2, 2, −2, 1/3, …`, reduced fractions only), a
documented revisitable choice exactly like ℤ's (round three, #17):

In [29]:
ℚ[3]

1:0: 1/2


1/2

In [30]:
X.cardinality()

1:0: ℵ₀


ℵ₀

## 8 · Functions

A function is `binder ↦ body` together with the domains it runs between, and
the two spellings SPEC.md uses — the lambda and `f(t) := …` — denote the
*same* function. `ℝ → ℝ` is an ascription **domain tag** at this stage: it
says where the function is declared and attaches no analysis semantics.

Bodies are exact polynomials, which is what lets the assertions below be
identities of function *expressions* rather than samples at a few points.
Non-polynomial bodies (`t ↦ sin(t)`, `t ↦ e^t`) are an honest gap until the
calculus sections land — they are refused at the binding, never
approximated.

In [31]:
let h := t ↦ t² + 1 in ℝ → ℝ

1:0: h := t ↦ t^2 + 1 ∈ ℝ → ℝ


In [32]:
let hp(t) := t^2 + 1 in R->R

1:0: hp := t ↦ t^2 + 1 ∈ ℝ → ℝ


In [33]:
assert h = hp

1:0: ✓ h = hp


In [34]:
assert h(0) = 1

1:0: ✓ h(0) = 1


In [35]:
assert h(3) = 10

1:0: ✓ h(3) = 10


`h(-t) = h(t)` is not two numeric samples that happened to agree: both
sides *substitute* a polynomial into the body, and the normal forms are
compared. A function in scope makes the binder it names available as that
indeterminate, which is how `t` can be written freely here.

In [36]:
assert h(-t) = h(t)

1:0: ✓ h(-t) = h(t)


The leading-ascription spelling declares the same thing. (`e.image()`
couples to set comprehensions and lands with them; the declaration itself is
what this section covers.)

In [37]:
let e: ℕ → ℕ := n ↦ 2n

1:0: e := n ↦ 2n ∈ ℕ → ℕ


Composition is the point of having functions at all: `f ∘ g` is a function
like any other, and the identity it satisfies is decided by substituting one
body into the other.

In [38]:
let f(t) = t^2 in RR->RR

1:0: f := t ↦ t^2 ∈ ℝ → ℝ


In [39]:
let g(t) = t^3 in RR->RR

1:0: g := t ↦ t^3 ∈ ℝ → ℝ


In [40]:
assert (f ∘ g)(t) = t^6

1:0: ✓ (f ∘ g)(t) = t^6


## 9 · Semantic availability is not computability

`det` makes sense for a square matrix over any commutative ring — the
category layer says so (`MatrixElems`), and no implementation hole is
allowed to redefine the mathematics. But the developer has only routed
matrices with entries in ℚ: over the field ℤ/5, `det` is *semantically*
available and not yet executable. The audit surface shows the hole as
structured backlog:

In [41]:
#capability_gaps

1:0:   representative    method        category              status
  [1,2;3,4] ∈ Mat₂(ℤ/5) det           MatrixElems(2, ℤ/5)   no route matches this presentation (1 registered for the method, none matching)
                                  available by: declared directly on MatrixElems(2, ℤ/5)
  [1,2;3,4] ∈ Mat₂(ℤ/5) inverse       MatrixElems(2, ℤ/5)   no route matches this presentation (1 registered for the method, none matching)
                                  available by: declared directly on MatrixElems(2, ℤ/5)
  26 method/representative pair(s) are implemented; 2 are backlog.


  representative    method        category              status
  [1,2;3,4] ∈ Mat₂(ℤ/5) det           MatrixElems(2, ℤ/5)   no route matches this presentation (1 registered for the method, none matching)
                                  available by: declared directly on MatrixElems(2, ℤ/5)
  [1,2;3,4] ∈ Mat₂(ℤ/5) inverse       MatrixElems(2, ℤ/5)   no route matches this presentation (1 registered for the method, none matching)
                                  available by: declared directly on MatrixElems(2, ℤ/5)
  26 method/representative pair(s) are implemented; 2 are backlog.

In [42]:
let A := [1, 2; 3, 4] in Mat₂(ℤ/5)

1:0: A := [1, 2; 3, 4] ∈ Mat₂(ℤ/5)


So the next cell **fails on purpose** — with a structured
`NoImplementation` capability gap naming the method, the receiver
category, the presentation, and the routes considered. Not a parse error,
not a type error, and not a silent lie. This failing cell is part of the
proof.


In [43]:
A.det()

LeanError: NoImplementation: 'det' is mathematically available here, but no registered route can execute it for this presentation.
  method:            det
  receiver category: MatrixElems(2, ℤ/5)
  presentation:      [1, 2; 3, 4] ∈ Mat₂(ℤ/5)
  semantic path:     declared directly on MatrixElems(2, ℤ/5)
  routes considered: 1
    - det for element of Mat(_, ℚ) → backend sage, op "mat_det_q", priority 0
This is a developer backlog item, not a narrowing of the mathematics: the method stays available on the category.